In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [ ]:
df_model = pd.read_csv(r'/Users/jmarc/ BU/Github_Repos/sports_betting/data/new_combined.csv')

In [2]:
def expected_value(p, o):
    EV = p * (o - 1) - (1 - p) * 1
    return EV 

def log_return_volatility(f, b, p):
    """
    Compute per-bet log-return volatility (sigma) and expected log return (mu).
    
    f : fraction of bankroll bet
    b : net odds (decimal odds - 1)
    p : probability of winning
    """
    r_win = np.log(1 + f * b)
    r_lose = np.log(1 - f)
    mu = p * r_win + (1 - p) * r_lose
    sigma2 = p * (r_win - mu)**2 + (1 - p) * (r_lose - mu)**2
    sigma = np.sqrt(sigma2)
    return sigma, mu

def expected_max_drawdown(sigma, N):
    """
    Heuristic for expected maximum drawdown over N bets
    """
    return sigma * np.sqrt(2 * np.log(N))

def scale_kelly_for_mdd(p, odds, f_full, N, max_drawdown, tol=1e-4):
    """
    Find the largest fraction of full Kelly that keeps expected MDD <= max_drawdown
    
    p : probability of winning
    odds : decimal odds
    f_full : full Kelly fraction (fraction of bankroll)
    N : number of bets
    max_drawdown : tolerable drawdown fraction (0 < max_drawdown < 1)
    tol : numerical tolerance for convergence
    """
    b = odds - 1
    # binary search between 0 and 1 (fraction of full Kelly)
    low, high = 0.0, 1.0
    best_fraction = 0.0
    
    while high - low > tol:
        k = (low + high) / 2
        f_trial = k * f_full
        sigma, mu = log_return_volatility(f_trial, b, p)
        mdd_est = expected_max_drawdown(sigma, N)
        
        if mdd_est <= max_drawdown:
            best_fraction = k  # this fraction is safe, try higher
            low = k
        else:
            high = k  # too aggressive, try lower
    return best_fraction * f_full



In [ ]:
def kelly_edge(p, fair_decimal):
    edge = p - (1/(fair_decimal))
    return edge

def parlay_top_ev(data, bankroll, top_n=2):
    
    df_top_n = data.sort_values(by='choice_ev', ascending=False).iloc[:top_n]

    parlay_win = True
    parlay_kelly = df_top_n['choice_fstar'].values.sum()
    net_odds = np.prod(df_top_n['choice_real_odds'])-1

    for _, row in df_top_n.iterrows():
        if row['winner'] != row['pred_winner']:
            parlay_win = False

    if parlay_win is True:
        profit = bankroll * parlay_kelly * net_odds
    else: 
        profit = bankroll * parlay_kelly * -1

    return profit 

def run_per_bet_scaling(choice_ev, choice_proba, unweighted_fstar, choice_fair_odds,
                         max_drawdown, bankroll, choice_real_odds, choice_idx, winner_col,
                          group, group_stats, N, group_profit):

    for i, (f_star, p, fair_odds, real_odds, ev, bet_idx) in enumerate(zip(unweighted_fstar, choice_proba, choice_fair_odds, 
                                                                            choice_real_odds, choice_ev, choice_idx)):
        if f_star < 0 or ev < 0:                 
            profit = 0
            f_final = f_star
            net_odds = 0
        else:
            f_final = scale_kelly_for_mdd(p, fair_odds, f_star, N=N, max_drawdown=max_drawdown)  
            stake = bankroll * f_final
            winning_bet = True if int(group.iloc[i][winner_col]) == bet_idx else False 
            profit = stake * (real_odds - 1) if winning_bet else -stake
            net_odds = (real_odds - 1) if winning_bet else -1

        group_stats['choice_fstar'].append(f_final)
        group_stats['fight_payout'].append(profit)
        group_stats['net_odds'].append(net_odds)
        group_profit += profit

    return group_stats, group_profit


def simulate_kelly(df_val, df_final, prob_cols, fair_decimal_cols, real_decimal_cols,
                           pred_winner_col='pred_winner', winner_col='winner', date_col='date',
                           init_bankroll=1000, max_drawdown=.30, N=1000,
                            calc_parlay=False, test_other_ev = False):
                            
    bankroll = init_bankroll
    df_results = pd.DataFrame()
    df = df_final.sort_values(by=date_col)

    for date, group in df.groupby(date_col, sort=True):
        group = group.reset_index(drop=True)

        group_profit = 0
        unweighted_fstar = []
        group_stats = {'fight_payout':[], 'choice_fstar':[], 'net_odds':[]}
        choice_ev = []
        choice_proba = []
        choice_real_odds = []
        choice_fair_odds = []
        choice_idx = []

        for idx, row in group.iterrows():
            bet_idx = int(row[pred_winner_col])
            p = row[prob_cols[bet_idx]]
            fair_odds = row[fair_decimal_cols[bet_idx]]
            real_odds = row[real_decimal_cols[bet_idx]]
            ev = expected_value(p, real_odds)

            if ev <= 0 and test_other_ev is True: 
                new_bet = np.abs(bet_idx -1)
                ev = expected_value(1-p, row[real_decimal_cols[new_bet]])
                if ev > 0: 
                    bet_idx = new_bet
                    real_odds = row[real_decimal_cols[bet_idx]]
                    fair_odds = row[fair_decimal_cols[bet_idx]]
                    p = 1-p
                        
            kelly_default = kelly_edge(p, fair_odds) 
            unweighted_fstar.append(kelly_default)

        f_scaled, sigma_portfolio, sigma_portfolio_scaled, mu_portfolio, \
        sharpe_portfolio, sharpe_per_bet, sigma_per_bet, k, group_stats,\
        group_profit = run_per_bet_scaling(choice_ev, choice_proba, unweighted_fstar, choice_fair_odds, max_drawdown, 
                                            bankroll, choice_real_odds, choice_idx, winner_col, 
                                            group, group_stats, N, group_profit)

        parlay_net = 0
        if calc_parlay is True: 
            df_data = pd.DataFrame({'winner':group['winner'], 'pred_winner':group['pred_winner'], 'choice_ev':choice_ev,
                                    'choice_real_odds' : choice_real_odds, 'choice_fstar':group_stats['choice_fstar']})
            parlay_net = parlay_top_ev(df_data, bankroll, top_n=2)       
            
                          